# Caspase kinetics
Run from `notebooks/` with the latest six modules in `../functions/`.

Processes one model and one plot at a time. Selected models are resampled; all others load from disk. All plots and tables are saved. `DISPLAY_PLOTS` controls notebook previews only.

Start with a fresh kernel to release objects created by the previous notebook. `saved_plots` stores file paths, not plot objects. Full posterior objects are released after each model; tables and the small time-analysis dataframes remain available.

## 1. Settings

In [ ]:
import sys
import importlib
import gc
from io import BytesIO
from pathlib import Path

import numpy as np
import polars as pl
import pymc as pm
import xarray as xr
import sympy as sp
import matplotlib.pyplot as plt
from IPython.display import display, Image
from PIL import Image as PILImage

sys.path.insert(0, str(Path("../functions").resolve()))
import data_processing
import pymc_models
import plot
import tables as table_functions
import time_analysis
import time_plots

for module in [data_processing, pymc_models, plot, table_functions, time_analysis, time_plots]:
    importlib.reload(module)

RESAMPLE = False
DISPLAY_PLOTS = False
PNG_DPI = 300
PREVIEW_DPI = 80

# Model 6 is mildy concerning because it will crash your computer if you have not prepared low numbers for DRAWS and TUNE
# I ran model 6 at DRAW=100 and TUNE=100
# The rest should be fine as is (1-5)
MODELS_TO_RESAMPLE = [1, 2, 3]

RANDOM_SEED = 67
PRIOR_DRAWS = 100
DRAWS = 1000
TUNE = 1000
CHAINS = 4
CORES = 4
CUTOFF = 10
CONSECUTIVE = 2
ZOOM_PERCENT = 10
ZOOM_INTERVAL = 0.5
START_ZOOM = 0.75
END_ZOOM = 1.0

MODEL_DIR = Path("../model")
SVG_DIR = Path("../plots/svg")
PNG_DIR = Path("../plots/png")
TABLE_DIR = Path("../tables")
for folder in [MODEL_DIR, SVG_DIR, PNG_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
pl.Config.set_float_precision(6)

## 2. Data

In [ ]:
df_long = data_processing.load_data("../data/caspase.csv")
df_long.write_csv("../data/kinetics_data.csv")
df1, df2, df3, df4, df5, df6 = data_processing.prepare_model_frames(df_long)

model_frames = {1: df1, 2: df2, 3: df3, 4: df4, 5: df5, 6: df6}
C_B_initial2 = float(df3["Concentration_of_Product"][0])
saved_plots = {}
tables = {}
time_data = {}
offset_frames = []
kinetic_means = {}
dag_images = []

## 3. Save and release each figure

In [ ]:
def save_figure(figure, name):
    plot_type = name.split("_", 1)[1] if name.startswith("model") else name

    if name == "model_dags":
        plot_type = "dags"
    elif name.startswith("showcase"):
        plot_type = "showcase"
    elif plot_type.startswith("zoom_cut"):
        plot_type = "zoom_cut"
    elif plot_type.startswith("zoom_full"):
        plot_type = "zoom_full"

    svg_folder = SVG_DIR / plot_type
    png_folder = PNG_DIR / plot_type

    try:
        svg_folder.mkdir(parents=True, exist_ok=True)
        png_folder.mkdir(parents=True, exist_ok=True)

        svg_path = svg_folder / f"{name}.svg"
        png_path = png_folder / f"{name}.png"

        figure.savefig(svg_path, format="svg")
        figure.savefig(png_path, format="png", dpi=PNG_DPI)

        saved_plots[name] = {"svg": svg_path, "png": png_path}

        if DISPLAY_PLOTS:
            with BytesIO() as preview:
                figure.savefig(preview, format="png", dpi=PREVIEW_DPI)
                display(Image(data=preview.getvalue(), width=1000))
    finally:
        plt.close(figure)


def save_plot(plot_object, name):
    print(name)
    figure = plot_object.draw(show=False)
    save_figure(figure, name)
    del figure, plot_object
    gc.collect()

## 4. Fit or load, analyze, save, release
Only selected NetCDF files are overwritten. Posterior predictive measurements are generated one model at a time, even in load mode.

In [ ]:
for number in range(1, 7):
    name = f"model{number}"
    label = f"Model {number}"
    frame = model_frames[number]
    times, observed = data_processing.frame_arrays(frame)
    build = getattr(pymc_models, f"build_model{number}")
    if number in [1, 3, 4]:
        model = build(times, observed)
    elif number == 2:
        model = build(times, observed, C_B_initial2)
    else:
        _, levels, curve_times, _, _ = data_processing.joint_frame_arrays(frame)
        model = build(levels, curve_times, observed)
    substrate = frame["Initial_Conc_Substrate_uM"].to_numpy()
    prior_path = MODEL_DIR / f"model{number}_prior.nc"
    posterior_path = MODEL_DIR / f"model{number}_posterior.nc"

    if RESAMPLE and number in MODELS_TO_RESAMPLE:
        print(f"Resampling {label}")
        with model:
            prior = pm.sample_prior_predictive(draws=PRIOR_DRAWS, random_seed=RANDOM_SEED)
            idata = pm.sample(draws=DRAWS, tune=TUNE, chains=CHAINS, cores=CORES,
                              nuts_sampler="pymc", random_seed=RANDOM_SEED)
        prior.to_netcdf(prior_path, engine="h5netcdf")
        idata.to_netcdf(posterior_path, engine="h5netcdf")
    else:
        print(f"Loading {label}")
        with xr.open_datatree(prior_path, engine="h5netcdf") as saved:
            prior = saved.load()
        with xr.open_datatree(posterior_path, engine="h5netcdf") as saved:
            idata = saved.load()
        del saved

    tables[name] = table_functions.make_model_tables(idata, observed, substrate)
    table_functions.display_and_save_tables(tables[name], label, name, TABLE_DIR)

    dag = pm.model_to_graphviz(model)
    dag.attr(bgcolor="white", dpi="120")
    dag_images.append(dag.pipe(format="png"))
    del dag

    prior_data = plot.prepare_prior_data(prior, times, observed, substrate)
    save_plot(plot.plot_prior_check(prior_data, model_label=label), f"{name}_prior")
    del prior_data
    gc.collect()

    save_plot(plot.plot_posterior_predictive_zoom(
        model, idata, times, observed, substrate,
        random_seed=RANDOM_SEED, model_label=label, zoom=(number >= 5)), f"{name}_posterior_zoom")
    save_plot(plot.plot_residual_comparison(
        idata, times, observed, substrate, model_label=label), f"{name}_residuals")
    save_plot(plot.plot_joint_posterior(idata, model_label=label), f"{name}_joint_posterior")
    save_plot(plot.plot_prior_posterior_densities(
        prior, idata, model_label=label), f"{name}_parameter_densities")

    if number in [5, 6]:
        time_data[number] = time_analysis.prepare_time_data(frame, idata, hill=(number == 6))
        levels = frame["Initial_Conc_Substrate_uM"].unique().sort().to_numpy()
        offset_frames.append(pl.DataFrame({
            "C_A0": levels,
            f"b_{number}": idata["posterior"]["b"].sel(experiment=levels)
                .mean(dim=("chain", "draw")).values
        }))
        kinetic_means[number] = {
            "V_max": idata["posterior"]["V_max"].mean().item(),
            "K_m": idata["posterior"]["K_m"].mean().item(),
            "sigma": idata["posterior"]["sigma"].mean().item(),
            "C_B_start": dict(zip(levels, idata["posterior"]["C_B_start"].sel(experiment=levels)
                                  .mean(dim=("chain", "draw")).values)),
        }

    del model, prior, idata
    gc.collect()

## 5. All model DAGs
The grid uses images directly, without a dataframe row for every pixel. The SVG contains raster DAG images.

In [ ]:
saved_plots.update(
    plot.save_model_dags(dag_images, SVG_DIR, PNG_DIR, dpi=PNG_DPI)
)

if DISPLAY_PLOTS:
    display(Image(filename=str(saved_plots["model_dags"]["png"]), width=1000))

del dag_images
gc.collect()

## 6. Integrated equations

In [ ]:
C_A0, C_B, C_B0, V_max, K_m, h = sp.symbols("C_A0 C_B C_B0 V_max K_m h", positive=True)
t, tau = sp.symbols("t tau", nonnegative=True)
u, A, A_start = sp.symbols("u A A_start", positive=True)

display(sp.Eq(sp.Integral(V_max, (tau, 0, t)),
              sp.Integral(1 + K_m**h / (C_A0 - C_B)**h, (C_B, C_B0, C_B))))
integral = sp.integrate(u**(-h), (u, A, A_start))
hill_time = ((A_start - A + K_m**h * integral) / V_max).subs(
    {A: C_A0 - C_B, A_start: C_A0 - C_B0})
mm_time = (C_B - C_B0 - K_m * sp.log((C_A0 - C_B) / (C_A0 - C_B0))) / V_max
print("Michaelis–Menten")
display(sp.Eq(t, mm_time))
print("Hill, including h = 1")
display(sp.Eq(t, hill_time))

## 7. Time predictions and cutoff

In [ ]:
time_data_5 = time_data[5]
time_data_6 = time_data[6]
time_data_5_cut = time_analysis.cut_plateau(time_data_5, CUTOFF, CONSECUTIVE)
time_data_6_cut = time_analysis.cut_plateau(time_data_6, CUTOFF, CONSECUTIVE)

r_squared_5, r_squared_5_overall = time_analysis.time_r_squared(time_data_5)
r_squared_6, r_squared_6_overall = time_analysis.time_r_squared(time_data_6)
r_squared_5_by_group_cut, r_squared_5_cut = time_analysis.time_r_squared(time_data_5_cut)
r_squared_6_by_group_cut, r_squared_6_cut = time_analysis.time_r_squared(time_data_6_cut)
retained_rows_5 = time_analysis.retained_rows(time_data_5, time_data_5_cut)
retained_rows_6 = time_analysis.retained_rows(time_data_6, time_data_6_cut)

offsets = offset_frames[0].join(offset_frames[1], on="C_A0").sort("C_A0")
table_functions.display_table(offsets)
for name, counts, full_fit, cut_fit in [
    ("model5", retained_rows_5, r_squared_5, r_squared_5_by_group_cut),
    ("model6", retained_rows_6, r_squared_6, r_squared_6_by_group_cut)
]:
    print(name)
    for table in [counts, full_fit, cut_fit]:
        table_functions.display_table(table)

time_summary = pl.DataFrame({
    "Model": ["model5", "model6"],
    "Full time R²": [r_squared_5_overall, r_squared_6_overall],
    "Cut time R²": [r_squared_5_cut, r_squared_6_cut],
    "Finite full rows": [time_data_5.filter(pl.col("t").is_finite()).height,
                         time_data_6.filter(pl.col("t").is_finite()).height],
    "Finite cut rows": [time_data_5_cut.filter(pl.col("t").is_finite()).height,
                        time_data_6_cut.filter(pl.col("t").is_finite()).height]
})
table_functions.display_table(time_summary)

for table_name, table in [
    ("measurement_offsets", offsets),
    ("time_summary", time_summary),
    ("model5_time_r_squared_full", r_squared_5),
    ("model6_time_r_squared_full", r_squared_6),
    ("model5_time_r_squared_cut", r_squared_5_by_group_cut),
    ("model6_time_r_squared_cut", r_squared_6_by_group_cut),
    ("model5_retained_rows", retained_rows_5),
    ("model6_retained_rows", retained_rows_6),
    ("model5_time_data_full", time_data_5),
    ("model6_time_data_full", time_data_6),
    ("model5_time_data_cut", time_data_5_cut),
    ("model6_time_data_cut", time_data_6_cut),
]:
    table.write_csv(TABLE_DIR / f"{table_name}.csv")


## 8. Matching rows and final concentrations

In [ ]:
common_time_data = (
    time_data_5_cut.select("C_A0", "real_time", pl.col("t").alias("t_5"))
    .join(time_data_6_cut.select("C_A0", "real_time", pl.col("t").alias("t_6")),
          on=["C_A0", "real_time"], how="inner")
    .filter(pl.col("t_5").is_finite() & pl.col("t_6").is_finite())
)
common_r2 = []
for name in ["t_5", "t_6"]:
    _, score = time_analysis.time_r_squared(
        common_time_data.select("C_A0", "real_time", pl.col(name).alias("t")))
    common_r2.append(score)
matched_time_r_squared = pl.DataFrame({"Model": ["model5", "model6"],
                                       "Matched rows": [common_time_data.height] * 2,
                                       "Matched time R²": common_r2})
table_functions.display_table(matched_time_r_squared)

final_C_B_comparison = time_analysis.compare_final_C_B(
    time_data_5, time_data_5_cut, time_data_6, time_data_6_cut
)
table_functions.display_table(final_C_B_comparison)

for table_name, table in [
    ("matched_time_r_squared", matched_time_r_squared),
    ("matched_time_data", common_time_data),
    ("final_C_B_comparison", final_C_B_comparison),
]:
    table.write_csv(TABLE_DIR / f"{table_name}.csv")


## 9. Time plots, saved one at a time

In [ ]:
for number, full, cut in [
    (5, time_data_5, time_data_5_cut),
    (6, time_data_6, time_data_6_cut)
]:
    label = f"Model {number}"
    for suffix, data in [("full", full), ("cut", cut)]:
        save_plot(time_plots.plot_time_overall(data, f"{label}: {suffix.title()} Data"),
                  f"model{number}_time_{suffix}")
        end_time = data["real_time"].max()
        for i, start in enumerate(np.arange(0, end_time, ZOOM_INTERVAL)):
            save_plot(time_plots.plot_time_zoom(
                data, start, min(start + ZOOM_INTERVAL, end_time),
                ZOOM_PERCENT, model_label=label), f"model{number}_zoom_{suffix}_{i}")
    save_plot(time_plots.plot_time_zoom(
        cut, START_ZOOM, END_ZOOM, ZOOM_PERCENT, model_label=label),
        f"model{number}_zoom_cut_selected")

# Illustrate why time predictions fail near the plateau: t(C_B) for the
# 3.125 µM experiment with Model 5's posterior means.
SENSITIVITY_C_A0 = 3.125
save_plot(time_plots.plot_time_sensitivity(
    V_max=kinetic_means[5]["V_max"],
    K_m=kinetic_means[5]["K_m"],
    C_A0=SENSITIVITY_C_A0,
    C_B0=kinetic_means[5]["C_B_start"][SENSITIVITY_C_A0],
    sigma=kinetic_means[5]["sigma"],
    model_label="Model 5"), "model5_time_sensitivity")

# Showcase plot for the TL;DR and sharing: the after-cutoff time plot with plain wording.
save_plot(time_plots.plot_tldr(time_data_5_cut), "showcase_model5_time_cut")


Tables remain accessible through `tables["model5"]["estimates"]`, `["diagnostics"]`, `["fit"]`, and the integer `["divergences"]`. Plot files are in `saved_plots["model5_residuals"]["png"]`. Full posterior objects and plot objects are not retained. Setting `DISPLAY_PLOTS = False` avoids storing image previews in notebook output, but all files are still saved.